# 30×30 아키타입 유사도 행렬 M 빌드

오프라인 1회 실행 노트북. `model/CLAUDE.md` "M 빌드 단계" 1~6 구현.

1. **세그멘테이션 크롭** (`mattmdjaga/segformer_b2_clothes`) — 배경·얼굴·피부 제거
   - 검수용: RGBA (배경 투명, 원본 해상도)
   - 임베딩 입력용: **흰 배경(255,255,255) RGB** + 옷 마스크 bbox 타이트 크롭 + 정사각 패딩
   - 흰 배경 이유: FashionCLIP이 흰 배경 상품 이미지로 학습됨 + 올블랙 스타일이 검정/투명에 묻힘
2. **임베딩** (`patrickjohncyh/fashion-clip`) → **L2 정규화**
3. **아키타입별 평균** → **재정규화** = 앵커
4. **30×30 코사인** (이미지 없는 id는 NaN)
5. **off-diagonal 재스케일** (min→0, max→1, 대각선 1) — ⚠ **풀 커버리지에서만 의미 있음**. 부분 커버리지 실행에선 `M_raw`만 신뢰.
6. **검수 리포트** + `M.json` + `anchors.npy` 저장

부분 커버리지(예: 현재 id 13, 27만)에서도 에러 없이 돌고 missing id를 명확히 리포트.


## 1. 파라미터


In [ ]:
from pathlib import Path

# Colab 커널은 Mac 로컬 파일 못 봄 → Drive 경로 사용.
# 로컬 → Drive로 미리 동기화: Mac의 ~/Library/CloudStorage/GoogleDrive-<account>/My Drive/fitting/
DRIVE_ROOT       = Path("/content/drive/MyDrive/fitting")

ANCHOR_DIR       = Path("/content/drive/MyDrive/Fitting 이미지 데이터 모음")  # shortcut to "Shared with me"
CROP_DIR         = DRIVE_ROOT / "crops"
ARCHETYPES_JSON  = DRIVE_ROOT / "data/archetypes.json"
KEYWORDS_JSON    = DRIVE_ROOT / "data/keywords.json"     # 추후 keyword_sim용
OUT_M            = DRIVE_ROOT / "artifacts/M.json"
OUT_ANCHORS      = DRIVE_ROOT / "artifacts/anchors.npy"
OUT_ANCHORS_IDS  = DRIVE_ROOT / "artifacts/anchors_ids.json"
OUT_REPORT       = DRIVE_ROOT / "artifacts/report.md"

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp"}
N_ARCHETYPES = 30

# segformer 18-class 중 옷으로 간주할 라벨
# 0 BG, 1 Hat, 2 Hair, 3 Sunglasses, 4 Upper, 5 Skirt, 6 Pants, 7 Dress,
# 8 Belt, 9-10 Shoes, 11 Face, 12-15 Legs/Arms, 16 Bag, 17 Scarf
GARMENT_LABELS = {1, 4, 5, 6, 7, 8, 9, 10, 16, 17}

# 임베딩 입력 정사각 크기 (FashionCLIP 224 입력에 여유)
CROP_OUTPUT_SIZE = 384


## 2. Colab 셋업 (drive mount + 의존성)


In [ ]:
# Colab 커널이면 drive mount + pip install. (VS Code+Colab 환경: 로컬 파일은 별도 설정 없이 접근됨.)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !pip install -q transformers torch pillow opencv-python-headless numpy
    KERNEL_ENV = "colab"
except ImportError:
    KERNEL_ENV = "local"
    print("Colab 커널 아님 — drive mount/pip install 건너뜀")

import torch
if torch.cuda.is_available():
    DEVICE = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon
else:
    DEVICE = "cpu"
print(f"kernel={KERNEL_ENV}  device={DEVICE}")

# 빠른 sanity check
print(f"\nARCHETYPES_JSON exists: {ARCHETYPES_JSON.exists()}  ({ARCHETYPES_JSON})")
print(f"ANCHOR_DIR     exists: {ANCHOR_DIR.exists()}  ({ANCHOR_DIR})")


## 3. archetypes.json 로드 + 앵커 커버리지 스캔


In [ ]:
import json

archetypes = json.loads(ARCHETYPES_JSON.read_text(encoding="utf-8"))
assert len(archetypes) == N_ARCHETYPES, f"archetypes={len(archetypes)} expected {N_ARCHETYPES}"

ID_TO_NAME  = {a["id"]: a["name"]  for a in archetypes}
ID_TO_GENRE = {a["id"]: a["계열"]  for a in archetypes}

# 폴더명 = archetype id로 매핑
source_images = {}
for aid in range(1, N_ARCHETYPES + 1):
    folder = ANCHOR_DIR / str(aid)
    if not folder.is_dir():
        continue
    imgs = sorted(p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS)
    if imgs:
        source_images[aid] = imgs

present_ids = sorted(source_images.keys())
missing_ids = [aid for aid in range(1, N_ARCHETYPES + 1) if aid not in source_images]

print(f"=== 앵커 커버리지 ===")
print(f"present: {len(present_ids)}/{N_ARCHETYPES}")
for aid in present_ids:
    print(f"  id={aid:2d}  {ID_TO_NAME[aid]:<16}  ({len(source_images[aid])} imgs)")
print(f"\nmissing: {len(missing_ids)}/{N_ARCHETYPES}")
for aid in missing_ids:
    print(f"  id={aid:2d}  {ID_TO_NAME[aid]}")


## 4. 크롭

각 앵커 이미지에 대해 두 가지 산출:
- **embed/**: 흰 배경 RGB, 마스크 bbox 타이트 크롭 → 정사각 패딩 → CROP_OUTPUT_SIZE 리사이즈 (FashionCLIP 입력)
- **rgba/**: 배경 투명 RGBA 원본 해상도 (눈으로 검수용)


In [ ]:
import numpy as np
from PIL import Image
from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation

SEG_MODEL_ID = "mattmdjaga/segformer_b2_clothes"
print(f"loading {SEG_MODEL_ID} ...")
seg_processor = SegformerImageProcessor.from_pretrained(SEG_MODEL_ID)
seg_model     = SegformerForSemanticSegmentation.from_pretrained(SEG_MODEL_ID).to(DEVICE).eval()


def load_rgb(path: Path) -> Image.Image:
    img = Image.open(path)
    if img.mode in ("RGBA", "LA"):
        bg = Image.new("RGB", img.size, (255, 255, 255))
        bg.paste(img, mask=img.split()[-1])
        return bg
    return img.convert("RGB")


def segment_garment_mask(pil_img: Image.Image) -> np.ndarray:
    inputs = seg_processor(images=pil_img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        logits = seg_model(**inputs).logits  # (1, n_cls, h, w)
    upsampled = torch.nn.functional.interpolate(
        logits, size=pil_img.size[::-1], mode="bilinear", align_corners=False
    )
    pred = upsampled.argmax(dim=1)[0].cpu().numpy()
    return np.isin(pred, list(GARMENT_LABELS))


def crop_for_embed(pil_img: Image.Image, mask: np.ndarray) -> Image.Image:
    """흰 배경 합성 → bbox 타이트 크롭 → 정사각 패딩 → 리사이즈."""
    rgb = np.array(pil_img)
    white = np.full_like(rgb, 255)
    composite = np.where(mask[..., None], rgb, white).astype(np.uint8)

    ys, xs = np.where(mask)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    cropped = composite[y0:y1, x0:x1]
    h, w = cropped.shape[:2]

    side = max(h, w)
    pad_t = (side - h) // 2
    pad_b = side - h - pad_t
    pad_l = (side - w) // 2
    pad_r = side - w - pad_l
    padded = np.pad(
        cropped,
        ((pad_t, pad_b), (pad_l, pad_r), (0, 0)),
        mode="constant", constant_values=255,
    )
    return Image.fromarray(padded).resize(
        (CROP_OUTPUT_SIZE, CROP_OUTPUT_SIZE), Image.LANCZOS
    )


def crop_for_inspect(pil_img: Image.Image, mask: np.ndarray) -> Image.Image:
    """RGBA 검수본 (원본 해상도, 배경 투명)."""
    rgba = np.array(pil_img.convert("RGBA"))
    rgba[..., 3] = (mask * 255).astype(np.uint8)
    return Image.fromarray(rgba)


crop_results = {}
fail_count = 0
empty_mask_count = 0

for aid in present_ids:
    embed_dir = CROP_DIR / str(aid) / "embed"
    rgba_dir  = CROP_DIR / str(aid) / "rgba"
    embed_dir.mkdir(parents=True, exist_ok=True)
    rgba_dir.mkdir(parents=True, exist_ok=True)

    saved = []
    for src in source_images[aid]:
        try:
            pil = load_rgb(src)
            mask = segment_garment_mask(pil)
            if mask.sum() == 0:
                empty_mask_count += 1
                print(f"  ⚠ id={aid} {src.name}: empty mask, 스킵")
                continue
            embed_img = crop_for_embed(pil, mask)
            rgba_img  = crop_for_inspect(pil, mask)
            embed_path = embed_dir / f"{src.stem}.png"
            embed_img.save(embed_path)
            rgba_img.save(rgba_dir / f"{src.stem}.png")
            saved.append(embed_path)
        except Exception as e:
            fail_count += 1
            print(f"  ✗ id={aid} {src.name}: {e}")

    if saved:
        crop_results[aid] = saved
        print(f"id={aid:2d} {ID_TO_NAME[aid]:<16}  cropped {len(saved)}/{len(source_images[aid])}")

total_cropped = sum(len(v) for v in crop_results.values())
print(f"\n총 크롭 완료: {total_cropped} 장  |  빈 마스크: {empty_mask_count}  |  실패: {fail_count}")


## 5. FashionCLIP 임베딩 → L2 정규화 → 평균 → 재정규화


In [33]:
from transformers import CLIPModel, CLIPProcessor

FCLIP_MODEL_ID = "patrickjohncyh/fashion-clip"
print(f"loading {FCLIP_MODEL_ID} ...")
fclip_processor = CLIPProcessor.from_pretrained(FCLIP_MODEL_ID)
fclip_model     = CLIPModel.from_pretrained(FCLIP_MODEL_ID).to(DEVICE).eval()


def embed_image(path: Path) -> np.ndarray:
    """vision_model + visual_projection 직접 호출 (get_image_features 우회).
    일부 transformers/FashionCLIP 조합에서 get_image_features가 풀링 안 된
    (1, seq, hidden)을 반환하는 버그가 있어 명시적 경로 사용."""
    img = Image.open(path).convert("RGB")
    inputs = fclip_processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vision_out = fclip_model.vision_model(**inputs)
        pooled = vision_out.pooler_output            # (1, hidden)
        feats  = fclip_model.visual_projection(pooled)  # (1, projection_dim)
    v = feats[0].cpu().numpy().astype(np.float32)    # (D,)
    assert v.ndim == 1, f"unexpected shape {v.shape}"
    return v / (np.linalg.norm(v) + 1e-12)           # ① L2 정규화


anchors_present = {}
for aid in present_ids:
    if aid not in crop_results:
        continue
    vecs = []
    for path in crop_results[aid]:
        try:
            vecs.append(embed_image(path))
        except Exception as e:
            print(f"  ✗ id={aid} {path.name}: {e}")
    if not vecs:
        continue
    mean = np.mean(np.stack(vecs), axis=0)
    norm_before = float(np.linalg.norm(mean))
    anchors_present[aid] = mean / (norm_before + 1e-12)  # ② 평균 후 재정규화
    print(f"id={aid:2d} {ID_TO_NAME[aid]:<16}  n_img={len(vecs)}  mean_norm_before_renorm={norm_before:.4f}")

embed_present_ids = sorted(anchors_present.keys())
print(f"\n앵커 생성: {len(embed_present_ids)}/{N_ARCHETYPES}")


loading patrickjohncyh/fashion-clip ...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

id= 1 미니멀 시티보이          n_img=4  mean_norm_before_renorm=0.8516
id= 2 미니멀 시크걸           n_img=4  mean_norm_before_renorm=0.8767
id= 3 클린 베이직            n_img=4  mean_norm_before_renorm=0.9110
id= 4 모노톤 무드            n_img=4  mean_norm_before_renorm=0.8693
id= 5 쿨톤 미니멀            n_img=4  mean_norm_before_renorm=0.8711
id= 6 릴렉스 캐주얼           n_img=4  mean_norm_before_renorm=0.8414
id= 7 스포티 프레시           n_img=4  mean_norm_before_renorm=0.8462
id= 8 코지 무드             n_img=4  mean_norm_before_renorm=0.8820
id= 9 서프 캐주얼            n_img=4  mean_norm_before_renorm=0.8114
id=10 캠퍼스 프레피           n_img=4  mean_norm_before_renorm=0.8628
id=11 어반 스트릿            n_img=4  mean_norm_before_renorm=0.8284
id=12 하이엔드 스트릿          n_img=4  mean_norm_before_renorm=0.8906
id=13 테크웨어 무드           n_img=4  mean_norm_before_renorm=0.8893
id=14 스케이터 바이브          n_img=4  mean_norm_before_renorm=0.8473
id=15 모던 클래식            n_img=4  mean_norm_before_renorm=0.8757
id=16 소프트 클래식           n_img=4  mean_no

## 6. 30×30 코사인 (이미지 없는 id는 NaN)

`M[i][j]`에서 `i = archetype_id - 1` (0-indexed). 매칭 런타임에서 같은 인덱싱 사용.


In [34]:
M_raw = np.full((N_ARCHETYPES, N_ARCHETYPES), np.nan, dtype=np.float64)

# 대각선: present id만 1.0
for aid in embed_present_ids:
    M_raw[aid - 1, aid - 1] = 1.0

# off-diagonal: present 쌍만
for i, ai in enumerate(embed_present_ids):
    for aj in embed_present_ids[i + 1:]:
        cos = float(np.dot(anchors_present[ai], anchors_present[aj]))
        M_raw[ai - 1, aj - 1] = cos
        M_raw[aj - 1, ai - 1] = cos

embed_missing = [aid for aid in range(1, N_ARCHETYPES + 1) if aid not in anchors_present]
print(f"M_raw: {N_ARCHETYPES}×{N_ARCHETYPES}, present {len(embed_present_ids)}")
print(f"missing ids ({len(embed_missing)}): {embed_missing}")


M_raw: 30×30, present 30
missing ids (0): []


## 7. off-diagonal 재스케일 (min→0, max→1, 대각선 1)

⚠ **풀 커버리지에서만 의미 있음.** 부분 커버리지(off-diagonal 페어 부족, min≈max)에선 재스케일 스킵하고 `M_raw` 그대로 복사.


In [35]:
off_diag = []
n = N_ARCHETYPES
for i in range(n):
    for j in range(i + 1, n):
        v = M_raw[i, j]
        if not np.isnan(v):
            off_diag.append(v)

M_rescaled = M_raw.copy()
EPS = 1e-6
rescale_skipped = False

if len(off_diag) < 2:
    print(f"⚠ off-diagonal 페어 {len(off_diag)}개 — 재스케일 스킵 (M_raw 그대로)")
    rescale_skipped = True
else:
    omin, omax = float(np.min(off_diag)), float(np.max(off_diag))
    if omax - omin < EPS:
        print(f"⚠ off-diagonal min≈max ({omin:.4f}) — 재스케일 스킵")
        rescale_skipped = True
    else:
        print(f"off-diagonal raw 범위 [{omin:.4f}, {omax:.4f}] → [0, 1] 재스케일")
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue  # 대각선은 그대로
                v = M_raw[i, j]
                if not np.isnan(v):
                    M_rescaled[i, j] = (v - omin) / (omax - omin)

if rescale_skipped:
    print("→ 부분 커버리지: M_raw만 신뢰. M_rescaled는 raw 사본.")


off-diagonal raw 범위 [0.6841, 0.9243] → [0, 1] 재스케일


## 8. 검수 리포트

- present id별 최근접 top-5
- 시티보이(1) ↔ 시크걸(2) 거리
- 계열별 within / cross 평균 (같은 계열끼리 뭉치는지)


In [36]:
from datetime import datetime

TOP_K = 5
report_lines = []
def out(s=""):
    report_lines.append(s)
    print(s)

# ── 헤더 ───────────────────────────────────────────────────────
out("# M 빌드 검수 리포트")
out()
out(f"- 생성: {datetime.now().isoformat(timespec='seconds')}")
out(f"- device: {DEVICE}")
out(f"- 커버리지: present {len(embed_present_ids)}/{N_ARCHETYPES}, missing {len(embed_missing)}/{N_ARCHETYPES}")
out(f"- rescale_skipped: {rescale_skipped} (True면 M_raw만 신뢰)")
out()

# ── present / missing ──────────────────────────────────────────
out("## present ids")
for aid in embed_present_ids:
    n_imgs = len(crop_results.get(aid, []))
    out(f"- {aid} {ID_TO_NAME[aid]} ({ID_TO_GENRE[aid]}) — {n_imgs} imgs")
out()
out("## missing ids")
out(", ".join(str(a) for a in embed_missing) if embed_missing else "(없음)")
out()

# ── top-K 최근접 ───────────────────────────────────────────────
out(f"## 최근접 top-{TOP_K} (present-only)")
out()
for aid in embed_present_ids:
    out(f"### id={aid} {ID_TO_NAME[aid]} ({ID_TO_GENRE[aid]})")
    out()
    out("| rank | id | name | 계열 | cos |")
    out("|---|---|---|---|---|")
    row = M_raw[aid - 1].copy()
    row[aid - 1] = -np.inf
    row = np.where(np.isnan(row), -np.inf, row)
    order = np.argsort(-row)[:TOP_K]
    for rank, idx in enumerate(order, 1):
        nid = idx + 1
        v = M_raw[aid - 1, idx]
        if np.isneginf(v) or np.isnan(v):
            out(f"| {rank} | -- | -- | -- | (no data) |")
        else:
            out(f"| {rank} | {nid} | {ID_TO_NAME[nid]} | {ID_TO_GENRE[nid]} | {v:.4f} |")
    out()

# ── 남↔여 변형 ─────────────────────────────────────────────────
out("## 남↔여 변형 거리 (sanity)")
v12 = M_raw[0, 1]
if np.isnan(v12):
    out("- 시티보이(1) ↔ 시크걸(2): 데이터 없음 (id 1 또는 2 미커버)")
else:
    out(f"- 시티보이(1) ↔ 시크걸(2): cos = {v12:.4f}")
out()

# ── 계열별 within / cross ──────────────────────────────────────
out("## 계열별 within / cross 평균 (clustering check)")
out()
out("| 계열 | n_present | within | cross | diff |")
out("|---|---|---|---|---|")
for g in sorted(set(ID_TO_GENRE.values())):
    g_ids = [a for a in embed_present_ids if ID_TO_GENRE[a] == g]
    if len(g_ids) < 2:
        out(f"| {g} | {len(g_ids)} | n/a | n/a | n/a |")
        continue
    within = [M_raw[g_ids[i] - 1, g_ids[j] - 1]
              for i in range(len(g_ids)) for j in range(i + 1, len(g_ids))]
    cross  = [M_raw[ai - 1, aj - 1]
              for ai in g_ids for aj in embed_present_ids if ID_TO_GENRE[aj] != g]
    w = float(np.nanmean(within)) if within else float("nan")
    c = float(np.nanmean(cross))  if cross  else float("nan")
    out(f"| {g} | {len(g_ids)} | {w:.4f} | {c:.4f} | {w - c:+.4f} |")
out()

# ── off-diagonal 통계 + 극단 페어 ──────────────────────────────
out("## off-diagonal 통계 (raw)")
od_pairs = []
for i in range(N_ARCHETYPES):
    for j in range(i + 1, N_ARCHETYPES):
        v = M_raw[i, j]
        if not np.isnan(v):
            od_pairs.append((v, i + 1, j + 1))

if od_pairs:
    vals = [p[0] for p in od_pairs]
    out(f"- count: {len(vals)}")
    out(f"- min/max: {min(vals):.4f} / {max(vals):.4f}")
    out(f"- mean ± std: {float(np.mean(vals)):.4f} ± {float(np.std(vals)):.4f}")
    out()
    od_pairs.sort()
    out("**가장 먼 페어 (cos 낮음 = 다름)**")
    for v, ai, aj in od_pairs[:3]:
        out(f"- {ID_TO_NAME[ai]} ↔ {ID_TO_NAME[aj]}: {v:.4f}")
    out()
    out("**가장 가까운 페어 (cos 높음 = 비슷)**")
    for v, ai, aj in od_pairs[-3:][::-1]:
        out(f"- {ID_TO_NAME[ai]} ↔ {ID_TO_NAME[aj]}: {v:.4f}")
else:
    out("- (off-diagonal 페어 없음)")
out()

report_md = "\n".join(report_lines)
print(f"\n— report 빌드 완료 ({len(report_lines)} lines) —")


# M 빌드 검수 리포트

- 생성: 2026-06-10T04:36:14
- device: cuda
- 커버리지: present 30/30, missing 0/30
- rescale_skipped: False (True면 M_raw만 신뢰)

## present ids
- 1 미니멀 시티보이 (미니멀) — 4 imgs
- 2 미니멀 시크걸 (미니멀) — 4 imgs
- 3 클린 베이직 (미니멀) — 4 imgs
- 4 모노톤 무드 (미니멀) — 4 imgs
- 5 쿨톤 미니멀 (미니멀) — 4 imgs
- 6 릴렉스 캐주얼 (캐주얼) — 4 imgs
- 7 스포티 프레시 (캐주얼) — 4 imgs
- 8 코지 무드 (캐주얼) — 4 imgs
- 9 서프 캐주얼 (캐주얼) — 4 imgs
- 10 캠퍼스 프레피 (캐주얼) — 4 imgs
- 11 어반 스트릿 (스트릿) — 4 imgs
- 12 하이엔드 스트릿 (스트릿) — 4 imgs
- 13 테크웨어 무드 (스트릿) — 4 imgs
- 14 스케이터 바이브 (스트릿) — 4 imgs
- 15 모던 클래식 (클래식) — 4 imgs
- 16 소프트 클래식 (클래식) — 4 imgs
- 17 댄디 포멀 (클래식) — 4 imgs
- 18 비즈캐주얼 무드 (클래식) — 4 imgs
- 19 다크 아티스틱 (아티스틱) — 4 imgs
- 20 감성 내추럴 (아티스틱) — 4 imgs
- 21 빈티지 로맨틱 (아티스틱) — 4 imgs
- 22 뉴트로 무드 (아티스틱) — 4 imgs
- 23 유니크 크리에이터 (아티스틱) — 4 imgs
- 24 러블리 페미닌 (페미닌) — 4 imgs
- 25 걸크러시 시크 (페미닌) — 4 imgs
- 26 프렌치 시크 (페미닌) — 4 imgs
- 27 글램 맥시멀 (맥시멀) — 4 imgs
- 28 보헤미안 프리 (맥시멀) — 4 imgs
- 29 아메카지 무드 (맥시멀) — 4 imgs
- 30 컬러팝 무드 (맥시멀) — 4 imgs

## missing ids
(없음)



## 9. 저장 — M.json + anchors.npy


In [37]:
OUT_M.parent.mkdir(parents=True, exist_ok=True)

def matrix_to_jsonable(M):
    return [[None if np.isnan(v) else float(v) for v in row] for row in M]

result = {
    "version": "1.0",
    "n_archetypes": N_ARCHETYPES,
    "indexing": "M[i][j] where i = archetype_id - 1 (0-indexed)",
    "ids_present": embed_present_ids,
    "ids_missing": embed_missing,
    "rescale_skipped": rescale_skipped,
    "rescale_note": "M_rescaled is only meaningful at full coverage. With partial coverage, trust M_raw only.",
    "M_raw": matrix_to_jsonable(M_raw),
    "M_rescaled": matrix_to_jsonable(M_rescaled),
}
OUT_M.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"✓ M.json → {OUT_M}")

OUT_ANCHORS.parent.mkdir(parents=True, exist_ok=True)
anchors_array = np.stack([anchors_present[aid] for aid in embed_present_ids]).astype(np.float32)
np.save(OUT_ANCHORS, anchors_array)
OUT_ANCHORS_IDS.write_text(json.dumps(embed_present_ids), encoding="utf-8")
print(f"✓ anchors.npy → {OUT_ANCHORS}  shape={anchors_array.shape}")
print(f"✓ anchors_ids.json → {OUT_ANCHORS_IDS}  ids={embed_present_ids}")

OUT_REPORT.write_text(report_md, encoding="utf-8")
print(f"✓ report.md → {OUT_REPORT}  ({len(report_md)} chars)")


✓ M.json → /content/drive/MyDrive/fitting/artifacts/M.json
✓ anchors.npy → /content/drive/MyDrive/fitting/artifacts/anchors.npy  shape=(30, 512)
✓ anchors_ids.json → /content/drive/MyDrive/fitting/artifacts/anchors_ids.json  ids=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
✓ report.md → /content/drive/MyDrive/fitting/artifacts/report.md  (9508 chars)
